In [1]:
import tensorflow as tf
import numpy as np
import json
import os


In [2]:

# --- 1. Define the Prediction Function ---

def predict_skin_condition(image_path, model_path, class_names_path, img_width=224, img_height=224):
    """
    Loads a trained model and class names to predict the condition from a single image.

    Args:
        image_path (str): The full path to the image file.
        model_path (str): The full path to the saved .keras model file.
        class_names_path (str): The full path to the .json file with class names.
        img_width (int): The target width for the image.
        img_height (int): The target height for the image.

    Returns:
        tuple: A tuple containing the predicted class name (str) and the confidence (float).
    """
    # Check if files exist
    if not os.path.exists(model_path):
        return f"Error: Model file not found at {model_path}", 0.0
    if not os.path.exists(class_names_path):
        return f"Error: Class names file not found at {class_names_path}", 0.0
    if not os.path.exists(image_path):
        return f"Error: Image file not found at {image_path}", 0.0

    try:
        # Load the trained model
        model = tf.keras.models.load_model(model_path)

        # Load the class names
        with open(class_names_path, 'r') as f:
            class_names = json.load(f)

        # Load and preprocess the image
        image = tf.io.read_file(image_path)
        image = tf.image.decode_jpeg(image, channels=3)
        image = tf.image.resize(image, [img_width, img_height])
        image_batch = tf.expand_dims(image, 0) # Create a batch

        # The model expects preprocessed input. Use the ResNetV2 preprocessor.
        preprocessed_image = tf.keras.applications.resnet_v2.preprocess_input(image_batch)

        # Make prediction
        pred_probs = model.predict(preprocessed_image)

        # Get the predicted class index and confidence
        pred_index = np.argmax(pred_probs[0])
        confidence = np.max(pred_probs[0])

        # Get the predicted class name
        pred_class_name = class_names[pred_index]

        return pred_class_name, float(confidence)

    except Exception as e:
        return f"An error occurred: {e}", 0.0


if __name__ == '__main__':
    # --- Configuration ---
    # !!! IMPORTANT: Update these paths to where your files are located !!!
    MODEL_FILE_PATH = r'C:\Users\vaghe\OneDrive\Desktop\Ge Model\Ge_ResNet50V2_Model.keras'
    CLASS_NAMES_FILE_PATH = r'C:\Users\vaghe\OneDrive\Desktop\Ge Model\skin_disease_class_names.json'
    
    # !!! IMPORTANT: Replace this with the path to the image you want to test !!!
    IMAGE_TO_PREDICT_PATH = r'C:\Users\vaghe\OneDrive\Desktop\Ge Model\Rashes_11.jpg'

    # --- Make a Prediction ---
    predicted_class, confidence_score = predict_skin_condition(
        image_path=IMAGE_TO_PREDICT_PATH,
        model_path=MODEL_FILE_PATH,
        class_names_path=CLASS_NAMES_FILE_PATH
    )

    # --- Display the Result ---
    if "Error" in predicted_class:
        print(predicted_class)
    else:
        print(f"Prediction complete.")
        print(f"   -> Predicted Condition: {predicted_class}")
        print(f"   -> Confidence Score:  {confidence_score:.2%}")

c:\Users\vaghe\anaconda3\envs\py31113\Lib\site-packages\keras\src\saving\saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adamw', because it has 56 variables whereas the saved optimizer has 60 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step
Prediction complete.
   -> Predicted Condition: Rashes
   -> Confidence Score:  77.97%
